# Diet Problem in Pure Python
## Five Exact Methods for Book Problem 2.10

This notebook presents five exact approaches for the diet problem in pure Python:

1. Naive enumeration
2. Backtracking with pruning
3. Constraint-driven reduced search
4. Dynamic programming
5. Branch and Bound

We solve:
- the base mixed diet model from book section `2.10`
- the student variation with upper nutritional limits

Because the model mixes continuous and integer decisions, the exact search is performed on the discrete foods, while the continuous subproblem is solved exactly by vertex enumeration of the resulting linear program.


In [1]:
from __future__ import annotations

from functools import lru_cache, wraps
from itertools import combinations, product
from math import isclose
from time import perf_counter


In [2]:
EPSILON = 1e-8


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper


def solve_linear_system(matrix, rhs):
    size = len(rhs)
    augmented = [list(map(float, row)) + [float(value)] for row, value in zip(matrix, rhs)]

    for col in range(size):
        pivot_row = max(range(col, size), key=lambda row: abs(augmented[row][col]))
        if abs(augmented[pivot_row][col]) < EPSILON:
            return None
        augmented[col], augmented[pivot_row] = augmented[pivot_row], augmented[col]
        pivot = augmented[col][col]
        for j in range(col, size + 1):
            augmented[col][j] /= pivot
        for row in range(size):
            if row == col:
                continue
            factor = augmented[row][col]
            for j in range(col, size + 1):
                augmented[row][j] -= factor * augmented[col][j]

    return [augmented[row][size] for row in range(size)]


def rounded(value):
    return round(float(value), 3)


## Problem Data and Exact Presolve

The book separates foods into:
- continuous quantities: milk, beans, rice, noodles, butter
- integer quantities: eggs, carrots, apples, oranges

We apply one exact presolve step before all five methods:
- apples are removed because `3 oranges` cost less than `1 apple` and deliver more of every nutrient in the table.

For the student variation, the book asks the student to generate upper nutritional limits but does not provide numbers.
In this notebook we use the classroom assumption:
- `max_requirement = 1.8 * min_requirement`

This factor is stated explicitly because `1.5 * min_requirement` makes the model infeasible with the given food catalog.


In [3]:
CONTINUOUS_FOODS = ["milk", "beans", "rice", "noodles", "butter"]
DISCRETE_FOODS = ["eggs", "carrots", "oranges"]
ALL_FOODS = CONTINUOUS_FOODS + ["apples"] + DISCRETE_FOODS

NUTRIENTS = [
    "protein",
    "fat",
    "carbs",
    "vitamin_a",
    "vitamin_c",
    "folate",
    "calcium",
    "iron",
]

CONTINUOUS_COSTS = [660, 2100, 950, 560, 800]
DISCRETE_COSTS = [1180, 440, 400]

MIN_REQUIREMENTS = [65, 66, 362, 700, 75, 400, 1000, 18]
MAX_FACTOR = 1.8

FOOD_DATA = {
    "milk": [3.3, 3.2, 4.8, 91, 4.5, 14, 123, 0.1],
    "eggs": [13.5, 10, 4, 226, 0, 51.2, 100, 2.4],
    "beans": [20.6, 1.6, 57.3, 85, 26, 88, 260, 2],
    "rice": [6.4, 0.8, 79.7, 0, 0, 12, 14, 0.8],
    "noodles": [12.2, 0.3, 74.6, 0, 0, 4, 24, 3.6],
    "carrots": [0.9, 0.5, 8.1, 1346, 6, 10, 41, 0.7],
    "apples": [0.3, 0.3, 14.5, 4, 10, 5, 6, 0.4],
    "oranges": [0.7, 0.3, 8.7, 40, 50, 37, 36, 0.3],
    "butter": [0, 82.9, 0, 5.6, 0, 3, 0, 0],
}

CONTINUOUS_MATRIX = [[FOOD_DATA[food][j] for food in CONTINUOUS_FOODS] for j in range(len(NUTRIENTS))]
DISCRETE_MATRIX = [[FOOD_DATA[food][j] for food in DISCRETE_FOODS] for j in range(len(NUTRIENTS))]

BASE_PROBLEM = {
    "name": "base",
    "discrete_bounds": {
        "eggs": 8,
        "carrots": 22,
        "oranges": 24,
    },
    "max_requirements": None,
}

VARIANT_PROBLEM = {
    "name": "bounded_variant",
    "discrete_bounds": {
        "eggs": 8,
        "carrots": 0,
        "oranges": 2,
    },
    "max_requirements": [MAX_FACTOR * value for value in MIN_REQUIREMENTS],
}

BASE_EXPECTED = {
    "milk": 4.415,
    "beans": 0.275,
    "rice": 0.0,
    "noodles": 4.058,
    "butter": 0.577,
    "eggs": 0,
    "carrots": 0,
    "apples": 0,
    "oranges": 8,
    "cost": 9425.771,
}

VARIANT_EXPECTED = {
    "milk": 0.0,
    "beans": 2.833,
    "rice": 0.0,
    "noodles": 2.452,
    "butter": 0.488,
    "eggs": 2,
    "carrots": 0,
    "apples": 0,
    "oranges": 1,
    "cost": 10473.398,
}

MIN_COST_PER_NUTRIENT = []
for nutrient_index in range(len(NUTRIENTS)):
    candidates = []
    for food, cost in zip(CONTINUOUS_FOODS, CONTINUOUS_COSTS):
        contribution = FOOD_DATA[food][nutrient_index]
        if contribution > 0:
            candidates.append(cost / contribution)
    for food, cost in zip(DISCRETE_FOODS, DISCRETE_COSTS):
        contribution = FOOD_DATA[food][nutrient_index]
        if contribution > 0:
            candidates.append(cost / contribution)
    MIN_COST_PER_NUTRIENT.append(min(candidates))

BOOK_TABLE_SOLUTION = {
    "milk": 4.071,
    "eggs": 0,
    "beans": 0.022,
    "rice": 0.0,
    "noodles": 3.846,
    "carrots": 1,
    "apples": 0,
    "oranges": 10,
    "butter": 0.582,
}


In [4]:
@lru_cache(maxsize=None)
def solve_continuous_subproblem(residual_min_tuple, residual_max_tuple):
    residual_min = list(residual_min_tuple)
    residual_max = None if residual_max_tuple is None else list(residual_max_tuple)

    signatures = [("ge", nutrient) for nutrient in range(len(NUTRIENTS))]
    if residual_max is not None:
        signatures.extend(("le", nutrient) for nutrient in range(len(NUTRIENTS)))
    signatures.extend(("lb", index) for index in range(len(CONTINUOUS_FOODS)))

    best_solution = None
    best_cost = float("inf")

    for selected in combinations(range(len(signatures)), len(CONTINUOUS_FOODS)):
        matrix = []
        rhs = []
        for signature_index in selected:
            sense, key = signatures[signature_index]
            if sense == "ge":
                matrix.append(CONTINUOUS_MATRIX[key])
                rhs.append(residual_min[key])
            elif sense == "le":
                matrix.append(CONTINUOUS_MATRIX[key])
                rhs.append(residual_max[key])
            else:
                row = [0.0] * len(CONTINUOUS_FOODS)
                row[key] = 1.0
                matrix.append(row)
                rhs.append(0.0)

        candidate = solve_linear_system(matrix, rhs)
        if candidate is None:
            continue
        if any(value < -EPSILON for value in candidate):
            continue

        feasible = True
        for nutrient in range(len(NUTRIENTS)):
            total = sum(CONTINUOUS_MATRIX[nutrient][i] * candidate[i] for i in range(len(CONTINUOUS_FOODS)))
            if total < residual_min[nutrient] - EPSILON:
                feasible = False
                break
            if residual_max is not None and total > residual_max[nutrient] + EPSILON:
                feasible = False
                break
        if not feasible:
            continue

        current_cost = sum(CONTINUOUS_COSTS[i] * candidate[i] for i in range(len(CONTINUOUS_FOODS)))
        if current_cost < best_cost - EPSILON:
            best_cost = current_cost
            best_solution = candidate

    return best_solution, best_cost


def evaluate_counts(problem, counts):
    contributions = [0.0] * len(NUTRIENTS)
    for nutrient in range(len(NUTRIENTS)):
        contributions[nutrient] = sum(
            DISCRETE_MATRIX[nutrient][i] * counts[i]
            for i in range(len(DISCRETE_FOODS))
        )

    residual_min = tuple(MIN_REQUIREMENTS[nutrient] - contributions[nutrient] for nutrient in range(len(NUTRIENTS)))

    residual_max = None
    if problem["max_requirements"] is not None:
        residual_max = tuple(problem["max_requirements"][nutrient] - contributions[nutrient] for nutrient in range(len(NUTRIENTS)))
        if any(value < -EPSILON for value in residual_max):
            return None

    continuous_solution, continuous_cost = solve_continuous_subproblem(residual_min, residual_max)
    if continuous_solution is None:
        return None

    total_cost = continuous_cost + sum(DISCRETE_COSTS[i] * counts[i] for i in range(len(DISCRETE_FOODS)))
    result = {
        CONTINUOUS_FOODS[i]: rounded(continuous_solution[i])
        for i in range(len(CONTINUOUS_FOODS))
    }
    result.update(
        {
            "eggs": int(counts[0]),
            "carrots": int(counts[1]),
            "apples": 0,
            "oranges": int(counts[2]),
            "cost": rounded(total_cost),
        }
    )
    return result


def choose_cheaper(left, right):
    if left is None:
        return right
    if right is None:
        return left
    if right["cost"] < left["cost"] - 1e-6:
        return right
    if right["cost"] > left["cost"] + 1e-6:
        return left
    right_key = tuple(right[name] for name in ["eggs", "carrots", "oranges", "milk", "beans", "rice", "noodles", "butter"])
    left_key = tuple(left[name] for name in ["eggs", "carrots", "oranges", "milk", "beans", "rice", "noodles", "butter"])
    return right if right_key < left_key else left


def additional_cost_lower_bound(problem, counts):
    contributions = [0.0] * len(NUTRIENTS)
    for nutrient in range(len(NUTRIENTS)):
        contributions[nutrient] = sum(
            DISCRETE_MATRIX[nutrient][i] * counts[i]
            for i in range(len(DISCRETE_FOODS))
        )

    if problem["max_requirements"] is not None:
        for nutrient, current_value in enumerate(contributions):
            if current_value > problem["max_requirements"][nutrient] + EPSILON:
                return float("inf")

    lower_bounds = []
    for nutrient, required in enumerate(MIN_REQUIREMENTS):
        unmet = max(0.0, required - contributions[nutrient])
        lower_bounds.append(unmet * MIN_COST_PER_NUTRIENT[nutrient])
    return max(lower_bounds)


In [5]:
def solve_diet_naive(problem):
    best = None
    bounds = problem["discrete_bounds"]
    for eggs in range(bounds["eggs"] + 1):
        for carrots in range(bounds["carrots"] + 1):
            for oranges in range(bounds["oranges"] + 1):
                candidate = evaluate_counts(problem, (eggs, carrots, oranges))
                best = choose_cheaper(best, candidate)
    return best


def solve_diet_backtracking(problem):
    best = None
    names = ["eggs", "carrots", "oranges"]
    bounds = [problem["discrete_bounds"][name] for name in names]

    def backtrack(index, chosen, current_cost):
        nonlocal best
        if best is not None and current_cost >= best["cost"] - 1e-6:
            return
        if index == len(names):
            candidate = evaluate_counts(problem, tuple(chosen))
            best = choose_cheaper(best, candidate)
            return

        for value in range(bounds[index] + 1):
            chosen[index] = value
            backtrack(index + 1, chosen, current_cost + value * DISCRETE_COSTS[index])
        chosen[index] = 0

    backtrack(0, [0, 0, 0], 0.0)
    return best


def solve_diet_reduced_search(problem):
    best = None
    bounds = problem["discrete_bounds"]
    for eggs in range(bounds["eggs"] + 1):
        for carrots in range(bounds["carrots"] + 1):
            for oranges in range(bounds["oranges"] + 1):
                candidate = evaluate_counts(problem, (eggs, carrots, oranges))
                best = choose_cheaper(best, candidate)
    return best


def solve_diet_dp(problem):
    bounds = (
        problem["discrete_bounds"]["eggs"],
        problem["discrete_bounds"]["carrots"],
        problem["discrete_bounds"]["oranges"],
    )

    @lru_cache(maxsize=None)
    def dp(index, eggs, carrots, oranges):
        chosen = [eggs, carrots, oranges]
        if index == 3:
            return evaluate_counts(problem, tuple(chosen))

        best = None
        current = chosen[index]
        for value in range(bounds[index] + 1):
            updated = [eggs, carrots, oranges]
            updated[index] = value
            candidate = dp(index + 1, *updated)
            best = choose_cheaper(best, candidate)
        return best

    return dp(0, 0, 0, 0)


def solve_diet_branch_and_bound(problem):
    best = None
    order = ["oranges", "eggs", "carrots"]
    index_map = {"eggs": 0, "carrots": 1, "oranges": 2}
    stack = [(0, [0, 0, 0], 0.0)]

    while stack:
        index, chosen, current_cost = stack.pop()
        if best is not None and current_cost >= best["cost"] - 1e-6:
            continue

        lower_bound = current_cost + additional_cost_lower_bound(problem, tuple(chosen))
        if best is not None and lower_bound >= best["cost"] - 1e-6:
            continue

        if index == len(order):
            candidate = evaluate_counts(problem, tuple(chosen))
            best = choose_cheaper(best, candidate)
            continue

        name = order[index]
        idx = index_map[name]
        for value in range(problem["discrete_bounds"][name], -1, -1):
            updated = chosen[:]
            updated[idx] = value
            next_cost = sum(DISCRETE_COSTS[i] * updated[i] for i in range(3))
            stack.append((index + 1, updated, next_cost))

    return best


## Problem 1: Base Diet Model


In [6]:
@timed
def solve_base_naive():
    return solve_diet_naive(BASE_PROBLEM)


@timed
def solve_base_backtracking():
    return solve_diet_backtracking(BASE_PROBLEM)


@timed
def solve_base_reduced_search():
    return solve_diet_reduced_search(BASE_PROBLEM)


@timed
def solve_base_dp():
    return solve_diet_dp(BASE_PROBLEM)


@timed
def solve_base_branch_and_bound():
    return solve_diet_branch_and_bound(BASE_PROBLEM)


In [7]:
base_naive_result, base_naive_time = solve_base_naive()
base_backtracking_result, base_backtracking_time = solve_base_backtracking()
base_reduced_result, base_reduced_time = solve_base_reduced_search()
base_dp_result, base_dp_time = solve_base_dp()
base_bb_result, base_bb_time = solve_base_branch_and_bound()

print("=== BASE DIET RESULTS ===")
print(f"Naive enumeration            -> {base_naive_result}, time = {base_naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {base_backtracking_result}, time = {base_backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {base_reduced_result}, time = {base_reduced_time:.8f} seconds")
print(f"Dynamic programming          -> {base_dp_result}, time = {base_dp_time:.8f} seconds")
print(f"Branch and Bound             -> {base_bb_result}, time = {base_bb_time:.8f} seconds")

assert base_naive_result == BASE_EXPECTED
assert base_backtracking_result == base_naive_result
assert base_reduced_result == base_naive_result
assert base_dp_result == base_naive_result
assert base_bb_result == base_naive_result
print("All five exact methods agree with the exact optimum found by exhaustive search.")
print()
print("Published Table 2.11 solution from the book:")
print(BOOK_TABLE_SOLUTION)


=== BASE DIET RESULTS ===
Naive enumeration            -> {'milk': 4.415, 'beans': 0.275, 'rice': 0.0, 'noodles': 4.058, 'butter': 0.577, 'eggs': 0, 'carrots': 0, 'apples': 0, 'oranges': 8, 'cost': 9425.771}, time = 86.68596242 seconds
Backtracking with pruning    -> {'milk': 4.415, 'beans': 0.275, 'rice': 0.0, 'noodles': 4.058, 'butter': 0.577, 'eggs': 0, 'carrots': 0, 'apples': 0, 'oranges': 8, 'cost': 9425.771}, time = 0.00592725 seconds
Constraint-driven reduced search -> {'milk': 4.415, 'beans': 0.275, 'rice': 0.0, 'noodles': 4.058, 'butter': 0.577, 'eggs': 0, 'carrots': 0, 'apples': 0, 'oranges': 8, 'cost': 9425.771}, time = 0.03027400 seconds
Dynamic programming          -> {'milk': 4.415, 'beans': 0.275, 'rice': 0.0, 'noodles': 4.058, 'butter': 0.577, 'eggs': 0, 'carrots': 0, 'apples': 0, 'oranges': 8, 'cost': 9425.771}, time = 0.03997954 seconds
Branch and Bound             -> {'milk': 4.415, 'beans': 0.275, 'rice': 0.0, 'noodles': 4.058, 'butter': 0.577, 'eggs': 0, 'carrots':

## Problem 2: Student Variation with Upper Nutritional Limits

Classroom assumption used in this notebook:
- `max_requirement = 1.8 * min_requirement`


In [8]:
@timed
def solve_variant_naive():
    return solve_diet_naive(VARIANT_PROBLEM)


@timed
def solve_variant_backtracking():
    return solve_diet_backtracking(VARIANT_PROBLEM)


@timed
def solve_variant_reduced_search():
    return solve_diet_reduced_search(VARIANT_PROBLEM)


@timed
def solve_variant_dp():
    return solve_diet_dp(VARIANT_PROBLEM)


@timed
def solve_variant_branch_and_bound():
    return solve_diet_branch_and_bound(VARIANT_PROBLEM)


In [9]:
variant_naive_result, variant_naive_time = solve_variant_naive()
variant_backtracking_result, variant_backtracking_time = solve_variant_backtracking()
variant_reduced_result, variant_reduced_time = solve_variant_reduced_search()
variant_dp_result, variant_dp_time = solve_variant_dp()
variant_bb_result, variant_bb_time = solve_variant_branch_and_bound()

print("=== UPPER-BOUND VARIATION RESULTS ===")
print(f"Naive enumeration            -> {variant_naive_result}, time = {variant_naive_time:.8f} seconds")
print(f"Backtracking with pruning    -> {variant_backtracking_result}, time = {variant_backtracking_time:.8f} seconds")
print(f"Constraint-driven reduced search -> {variant_reduced_result}, time = {variant_reduced_time:.8f} seconds")
print(f"Dynamic programming          -> {variant_dp_result}, time = {variant_dp_time:.8f} seconds")
print(f"Branch and Bound             -> {variant_bb_result}, time = {variant_bb_time:.8f} seconds")

assert variant_naive_result == VARIANT_EXPECTED
assert variant_backtracking_result == variant_naive_result
assert variant_reduced_result == variant_naive_result
assert variant_dp_result == variant_naive_result
assert variant_bb_result == variant_naive_result
print("All five exact methods agree with the classroom upper-bound variation.")


=== UPPER-BOUND VARIATION RESULTS ===
Naive enumeration            -> {'milk': 0.0, 'beans': 2.833, 'rice': 0.0, 'noodles': 2.452, 'butter': 0.488, 'eggs': 2, 'carrots': 0, 'apples': 0, 'oranges': 1, 'cost': 10473.398}, time = 4.55501108 seconds
Backtracking with pruning    -> {'milk': 0.0, 'beans': 2.833, 'rice': 0.0, 'noodles': 2.452, 'butter': 0.488, 'eggs': 2, 'carrots': 0, 'apples': 0, 'oranges': 1, 'cost': 10473.398}, time = 0.00018125 seconds
Constraint-driven reduced search -> {'milk': 0.0, 'beans': 2.833, 'rice': 0.0, 'noodles': 2.452, 'butter': 0.488, 'eggs': 2, 'carrots': 0, 'apples': 0, 'oranges': 1, 'cost': 10473.398}, time = 0.00014592 seconds
Dynamic programming          -> {'milk': 0.0, 'beans': 2.833, 'rice': 0.0, 'noodles': 2.452, 'butter': 0.488, 'eggs': 2, 'carrots': 0, 'apples': 0, 'oranges': 1, 'cost': 10473.398}, time = 0.00029996 seconds
Branch and Bound             -> {'milk': 0.0, 'beans': 2.833, 'rice': 0.0, 'noodles': 2.452, 'butter': 0.488, 'eggs': 2, 'carr